# Sistema de Predicción de Riesgo biológico post-Radioterapia 

Este proyecto, desarrollado para la **Universidad del Valle de Guatemala**, utiliza una **Red Neuronal** y el estándar **OMOP CDM** para predecir el riesgo de efectos biológicos tardíos basados en la dosis acumulada de radiación.

## Requisitos
- sqlalchemy
- pandas
- psycopg2-binary
- scipy
- scikit-learn
- lifelines
- matplotlib
- seaborn
- numpy

## Instrucciones para entrar a jupyter
1. Clona este repositorio: `git clone https://github.com/TU_USUARIO/nombre-repositorio.git`
2. Dirigete a la carpeta del proyecto con `cd Nombre del Repositorio`
3. Crea una nueva rama, de esa manera si debes de realizar algún cambio no entrara en conflicto con la rama principal y sera revisado por uno de los administradores, usa  `git branch -a`
4. Define el nombre de tu rama y cambiate a ella con `git checkout -b nombre-de-tu-rama`
5. Añade los cambios al repositorio para que tu rama se cree en el repositorio con `git add .`
6. Realiza un commit para verificar que lo hiciste bien `git commit -m "Clonando repositorio"`
7. Sube los cambios al repositorio con `git push origin nombre-de-tu-rama`
8. Cambia a la rama principal con`git checkout main`
9. Manda a llamar los cambios a tu computadora, asi mandas a llamar el archivo necesario para levantar Docker `git pull origin main`
10. Una vez dentro verifica que te encuentras en la carpeta de este archivo, sino es asi transportate a ella usando el comando cd y path "docker-compose.yml"
11. Levanta los servicios estando en la ubicación del archivo con `docker-compose up -d` Para esto ya deberias de tener levantado Docker y tener los contenedores limpios. Si todo esta bien procede con el siguiente paso.
12. Accede a Jupyter en `http://localhost:8888` (Token: `uvg_2026`)

## Instrucciones para ejecutar| Item | Valor |
|---|---|
| Dataset | CCSS: Childhood Cancer Survivor dy CCSS. (2021). Radiotherapy with select doses: All cancers Any radiation Max dose to brain 1 Max dose to chest data from Jan2020 data freeze-all % based on denominator with non-missing values Data for patients with medical records abstraction and completed baseline qx, info in 1st 5 years N/% reflect those with direct radiation; Patients with TBI not included in other categories. In Childhood Cancer Survivor Study. https://ccss.stjude.org/content/dam/en_US/shared/ccss/documents/data/data-radiotherapy-select-doses.pdf)dy |
umero inicial de pacientesff ffffffff|e
 |
| Pregunta clínica | ¿Existe una relación estadísticamente significativa entre la dosis acumulada de radioterapia recibida por pacientes oncológicos y la presencia de niveles detectables de radiación o efectos biológicos posteriores al tratami nto durante el seguimiento clínico? |
| Decisión informada | Mantener en seguimiento prioritario a aquellos pacientes con alta probabilidad de recurrencia del cáncer debido a niveles acumulados de radiación, recomendando chequeos más frecuentes. Para pacientes con menor probabilidad, la frecuencia de seguimiento dependerá del riesgo estipor el modelo | elo.d
| o f Regresión Logística |ada |

## 0. Decisión clínica a apoyar
Context: Actualmente el tratamiento más eficaz y común para tratar el cancer es la radioterapia, la cual ataca a las celulas del area afectada y las va eliminando poco a poco. Si el paciente sobrevive los niveles de radiación y el cancer se muere antes de que la condición del paciente se vuelva peligrosa se considera un exito. Esto es bueno, pero lo que muchos no saben es que los niveles de radiación de la terapia no se eliminan inmediatamente, una parte de la radiación se queda en el cuerpo. Y así como ayudo a eliminar el cancer, esta puede mutar las celulas y traer nuevamente el cancér. De esto surge la necesidad de evaluar el riesgo que tienen los pacientes que tuvieron cancer, de recaer debido a los niveles de radiación residuales. Que tan necesario es que se hagan chequeos y con que regularid.
?#### 
Decisión que el modelo informa, no reelaza. az
- Alta probabilidad **NO** garantiza que el paciente incurrira en el cancer, ni debe de empujar a las personas a buscar tratamiento o remover areas afectadas en base a esto. Esto solamente indica que la acumulación de radiación es tal que se debe de considerar la posibilidad. Lo mismo aplica con baja probabilidad, si el doctor considera que el modelo esta equivocado, es mejor que siga su experiencia.
- **NO** reemplaza la opinion y experienca de un profesional de la salud, esto unicamente busca ayudar a los doctores a mantener en la periferia aquellos pacientes que necesitan un mayor monitoreo debido a su historial médico. A recomendarles chequeos frecuentes para atrapar cualquier problema antes de que se vuelva un complicación mayor.
- El modelo **NO es perfecto**, como cualquier modelo puede incurrir en errores y falsos positivos, por lo que igualmente no debe tomarse como escrito en piedra lo que diga.aos 0.5.

In [ ]:
import pandas as pd
import sqlalchemy
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sqlalchemy import text
from scipy.stats import pearsonr
from lifelines import KaplanMeierFitter
from sklearn.linear_model import LogisticRegression

## 1. Adquisición y Gobernanza
Fuente: CCSS: Childhood Cancer Survivor Study, dataset **Radiotherapy with select doses: All cancers**, procesado por la Cohorte de Expansión (a enero de 2021), la Cohorte Original y la Cohorte General del CCSS. Los datos datan del 12 de enero del 2020, .... pacientes con extracción de historial clinico y cuestionario basal completado, informacion de los primeros 5 años. El porcentaje de pacientes con radiación directa refleja aquellos que recibieron radiación directa; los pacientes con irradiación corporal total no se incluyen en otras categorías. El nivel de 1 a 19,9 Gy incluye dosis de radiación dispersa. Licencia: Dominio público (CCO). Citación obligatoria: CCSS. (2021). Radiotherapy with select doses: All cancers Any radiation Max dose to brain 1 Max dose to chest data from Jan2020 data freeze-all % based on denominator with non-missing values Data for patients with medical records abstraction and completed baseline qx, info in 1st 5 years N/% reflect those with direct radiation; Patients with TBI not included in other categories. In Childhood Cancer Survivor Study. https://ccss.stjude.org/content/dam/en_US/shared/ccss/documents/data/data-radiotherapy-select-doses.pdf

Uso secundario de datos, su uso original era recabar las características demográficas, del cáncer primario y del tratamiento de la Cohorte de Expansión (a enero de 2021), la Cohorte Original y la Cohorte General del CCSS.

Estos datos están disponibles como recurso para investigadores. El acceso a estos datos comienza con la presentación de una Solicitud de Intención (AOI), seguida del desarrollo de una Propuesta Formal de Concepto de Análisis que deberá ser aprobada por el Comité de Publicaciones del CCS Para propositos de este proyecto, puesto que no esta pensado en ser utilizado por profesionales, se puede utilizar la base de datos púlica disponible.

Tipo de datos organizado puesto que la información se encuentra en tablas utilizables organizadas por filas y columnas.

Consideraciones Éticas: Los datos ya se encuentran anonimizados por lo que no existe riesgo de identificación, y puesto que la base de datos es pública y no sera utilizada para investigación (restricción de uso), no se necesita consentimiento para utilizarla en este proyeo.ct#### # MAPEO A ESTANDARES SEMANTI** 
El proyecto utiliza una estructura inspirada en OMOP-CDM mediante las tablas:

- person
- procedure_occurrence
- measurement

Estas tablas permiten almacenar información demográfica, procedimientos de radioterapia y mediciones clínicas relacionadas con dosis y efectos biológnte lo permite?
.

, 303 pacientes con sospecha de enfermedad coronaria evaluados con estud
i**nvasivos..i

onb

odENENQ UE V: **Los datos clínicos son almacenados y procesados utilizando PostgreSQL mediante SQLAlchemy.
La información es organizada en tablas estructuradas para permitir trazabilidad, análisis estadístico y reconstrucción de resultado 

El procesamiento de los datos, cálculo de riesgo y análisis estadístico se realizan directamente dentro del notebivo.ook.onstruirse desde el CSV.

In [ ]:
import sqlalchemy
import pandas as pd
from sqlalchemy import text

# Conexión a la base de datos PostgreSQL
engine = sqlalchemy.create_engine(
    'postgresql://ivon_user:uvg_password@db_radioterapia:5432/omop_radioterapia'
)

# Verificar conexión
try:
    with engine.connect() as connection:
        print("¡Conexión exitosa!")
except Exception as e:
    print(f"Error de conexión: {e}")

# Inserción de datos clínicos de prueba
sql_insertar_datos = """
INSERT INTO person (person_id, gender_concept_id, year_of_birth, enfermedades_previas) 
VALUES (1, 8532, 1970, 'Diabetes'), (2, 8507, 2005, 'Ninguna')
ON CONFLICT DO NOTHING;

INSERT INTO procedure_occurrence (person_id, tecnica_rt, region_irradiada, num_fracciones, fecha_inicio)
VALUES (1, 'IMRT', 'Tórax', 30, '2025-01-01'),
       (2, '3D-CRT', 'Pelvis', 25, '2025-02-01')
ON CONFLICT DO NOTHING;

INSERT INTO measurement (person_id, tipo_medicion, valor_numerico, fecha_medicion)
VALUES (1, 'Dosis Total', 60.0, '2025-02-15'),
       (2, 'Dosis Total', 45.0, '2025-03-01')
ON CONFLICT DO NOTHING;
"""

with engine.connect() as connection:
    connection.execute(text(sql_insertar_datos))
    connection.commit()
    print("Datos clínicos insertados correctamente.")

## 2. Preparación y Curación

- Los datos clínicos fueron organizados y procesados utilizando tablas inspiradas en OMOP-CDM: person, procedure_occurrence y measurement.
- Se realizó una integración de las tablas mediante operaciones JOIN para consolidar información demográfica, tratamientos de radioterapia y mediciones dosimétricas dentro de un único DataFrame clínico.
- Las transformaciones y variables derivadas fueron documentadas directamente dentro del notebook para asegurar trazabilidad y reproducibilidad del analisis.
- Se verificó la coherencia entre pacientes, procedimientos y mediciones mediante relaciones por person_id
- 
También se validó la consistencia temporal entre fechas de tratamiento y mediciones clínicas, así como la prsencia dee valores numéricos válidos para dosis acumuladas de radiotrapia.i
-.El notebook no implementa un procedimiento formal de imputación de valores faltntess 
Los registros utilizados en el análisis corresponden únicamente a pacientes con información disponible para las variables necesarias en el cálculo de riesgo y análisis esdístico
co
Se generaron variables derivadas clínicamente relevantes, incluyendo:

- Edad del paciente calculada a partir del año de nacimiento.
- Dosis total acumulada en Gy.
- Porcentaje de riesgo biológico calculado mediante una función heurística basada en dosis, edad, comorbilidades y tratamientos adicionales.
- Clasificación automática de frecuencia de seguimiento clínico según nivel de riesgo.

Estas transformaciones permitieron construir variables predictoras y variables objetivo para los modelos estmachine learning.

maninne lear**Decisiones explícitas tomadas durante la curación y preparació datos:**

• Se seleccionaron únicamente mediciones con tipo_medicion = 'Dosis Total' para asegurar consistencia en el análisis dosimétrico.

• La edad fue derivada a partir del año de nacimiento para facilitar el modelado clínico.

• El porcentaje de riesgo biológico fue limitado a un máximo de 100% mediante normalización.

• La variable de seguimiento clínico fue categorizada en tres niveles:
    - Intensivo: ≥70%
    - Moderado: 40%–69%
    - Anual: <40%

• Para el modelo de regresión logística, la toxicidad severa fue simulada utilizando:
    y = 1 si el porcentaje de riesgo > 60%
    y = 0 en caso contrario.

Esto permitió construir un modelo predictivo supervisado utilizando variables clínicas y dosimétricas.line de scikit-learn.

In [ ]:
#Tablas inspiradas en OMOP-CDM
CREATE TABLE person (
    person_id INT PRIMARY KEY,
    gender_concept_id INT,
    year_of_birth INT,
    enfermedades_previas TEXT,
    tratamientos_adicionales TEXT
);

CREATE TABLE procedure_occurrence (
    procedure_id SERIAL PRIMARY KEY,
    person_id INT REFERENCES person(person_id),
    tecnica_rt TEXT,
    region_irradiada TEXT,
    num_fracciones INT,
    fecha_inicio DATE,
    fecha_fin DATE
);

CREATE TABLE measurement (
    measurement_id SERIAL PRIMARY KEY,
    person_id INT REFERENCES person(person_id),
    tipo_medicion TEXT,
    valor_numerico FLOAT,
    fecha_medicion DATE
);

In [ ]:
#Integracion mediante JOIN
query_analisis = """
SELECT 
    p.person_id,
    (2026 - p.year_of_birth) as edad,
    p.enfermedades_previas,
    p.tratamientos_adicionales,
    pr.num_fracciones,
    pr.tecnica_rt,
    m.valor_numerico as dosis_total_gy
FROM person p
JOIN procedure_occurrence pr ON p.person_id = pr.person_id
JOIN measurement m ON p.person_id = m.person_id
WHERE m.tipo_medicion = 'Dosis Total';
"""

In [ ]:
#Construccion del dataframe clínico
df_clinico = pd.read_sql(query_analisis, engine)

In [ ]:
#Relaciones mediante person id
JOIN procedure_occurrence pr ON p.person_id = pr.person_id
JOIN measurement m ON p.person_id = m.person_id

person_id INT REFERENCES person(person_id)

In [ ]:
#Validacion temporal y variables clinicas
fecha_inicio DATE,
fecha_fin DATE,
fecha_medicion DATE

valor_numerico FLOAT

In [ ]:
#Edad derivada desde año de nacimiento
(2026 - p.year_of_birth) as edad
#Dosis total acumulada
m.valor_numerico as dosis_total_gy
WHERE m.tipo_medicion = 'Dosis Total';
#Calculo del porcentaje de riego
def calcular_sistema_riesgo(row):

    riesgo = 0

    riesgo += (row['dosis_total_gy'] * 0.8)

    if row['edad'] > 50:
        riesgo += 10

    if row['enfermedades_previas'] != 'Ninguna':
        riesgo += 15

    if row['tratamientos_adicionales'] == 'Quimioterapia':
        riesgo += 10

    porcentaje_final = min(riesgo, 100)

    if porcentaje_final >= 70:
        seguimiento = "Cada 3 meses (Intensivo)"

    elif 40 <= porcentaje_final < 70:
        seguimiento = "Cada 6 meses (Moderado)"

    else:
        seguimiento = "Cada 12 meses (Anual)"

    return pd.Series([porcentaje_final, seguimiento])

#Aplicacion del calculo al dataframe
df_clinico[['Porcentaje_Riesgo', 'Frecuencia_Consulta']] = df_clinico.apply(calcular_sistema_riesgo, axis=1)
#Normalizacion al maximo
porcentaje_final = min(riesgo, 100)
#Clasificacion de seguimiento clinico
if porcentaje_final >= 70:
    seguimiento = "Cada 3 meses (Intensivo)"

elif 40 <= porcentaje_final < 70:
    seguimiento = "Cada 6 meses (Moderado)"

else:
    seguimiento = "Cada 12 meses (Anual)"

#Construccion de variables predictoras para machine learning
X = df_clinico[['dosis_total_gy', 'edad']].values
#Construccion de variable objetivo
y = (df_clinico['Porcentaje_Riesgo'] > 60).astype(int)
#Entrenamiento del modelo supervisado
modelo_logistico = LogisticRegression()
modelo_logistico.fit(X, y)
#Visualizacion del dataset clinico
display(df_clinico[['person_id', 'edad', 'dosis_total_gy', 'Porcentaje_Riesgo', 'Frecuencia_Consulta']])

## 3. Analisis Exploratorio

Cada gráfico tiene un propósito para no cargar al usuario con información extra.

#### 3.1 Tabla clínica integrada
Construye un DataFrame clínico consolidado integrando:

- Datos demográficos
- Tratamientos de radioterapia
- Dosis acumuladas.

mediante operaciones JOIN entre tablas SQL.

In [ ]:
df_clinico = pd.read_sql(query_analisis, engine)

#### 3.2 Tabla de riesgo
Muestra el resultado final del sistema heurístico de riesgo biológico.

Incluye:

- ID del paciente
- Edad
- Dosis acumulada
- Porcentaje de riesgo
- Frecuencia recomendada de seguimiento clínico

In [ ]:
display(df_clinico[['person_id', 'edad', 'dosis_total_gy', 'Porcentaje_Riesgo', 'Frecuencia_Consulta']])

#### 3.3 Curva de supervivencia Kaplan-Meier (primera)
Genera una curva de supervivencia utilizando lifelines para estimar la probabilidad de permanecer libre de efectos biológicos tardíos después de la radioterapia.

Representa:

- Tiempo de seguimiento
- Ocurrencia de toxicidad
- Supervivencia libre de eventos

In [ ]:
kmf.plot_survival_function()

#### 3.4 Gráfico de correlación Dosis vs Riesgo
Genera un gráfico de dispersión con línea de regresión para visualizar la relación entre:

- Dosis total de radioterapia
- Porcentaje de riesgo biológico.

Permite validar visualmente la hipótesis clínica: a mayor dosis acumulada, mayor riesgo biológico

In [ ]:
sns.regplot(
    x='dosis_total_gy',
    y='Porcentaje_Riesgo',
    data=df_clinico,
    color='red'
)

#### 3.5 Resultado estadístico de Pearson
Calcula estadísticamente la fuerza de asociación entre:

- Dosis acumulada
- Riesgo biológico

El coeficiente de Pearson mide:

- Correlación positiva
- Negativa
- O ausencia de relación

El valor p evalúa significancia estadística.

In [ ]:
coef, p_val = pearsonr(
    df_clinico['dosis_total_gy'],
    df_clinico['Porcentaje_Riesgo']
)

#Y
print(f"Coeficiente de Pearson: {coef:.4f}")
print(f"Valor p: {p_val:.4f}")

#### 3.6 Curva Kaplan-Meier final (UVG Project)
Genera una segunda curva Kaplan-Meier incluyendo:

- Pacientes en riesgo
- Seguimiento post-radioterapia
- Probabilidad acumulada de permanecer libre de toxicidad

In [ ]:
kmf.plot_survival_function(at_risk_counts=True)